# Predicción de Pedidos Cancelados: Del Análisis de Datos al Modelado

Para este notebook, lo que haremos sera la creación de los modelos de clasificación para predecir si un pedido sera cancelado o entregado, basandonos en el análisis exploratorio de datos realizado en el notebook anterior.

## Librerías, configuraciones y carga de datos

In [ ]:
# Transformación y preparación de datos
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from collections import Counter
from imblearn.over_sampling import SMOTE

# Modelado y evaluación
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report, confusion_matrix, roc_auc_score, f1_score
from src.functions import evaluar_y_guardar_csv

# Se establece el número máximo de núcleos de CPU que Loky puede utilizar por motivos de rendimiento y evitar sobrecarga
os.environ['LOKY_MAX_CPU_COUNT'] = '8'

In [ ]:
# Configuración de colores y estilos de las graficas
color_entregado = "#4E79A7"
color_cancelado = "#E15759"
mis_colores_multi = ["#4E79A7", "#E15759", "#76B7B2", "#59A14F", "#EDC948", "#B07AA1", "#FF9DA7", "#9C755F"]

sns.set_theme(style="whitegrid", rc={"axes.facecolor": "#FBFBFB", "grid.color": "#ECECEC"})
sns.set_palette(sns.color_palette(mis_colores_multi))

order_status_palette = {'Delivered': color_entregado, 'Cancelled': color_cancelado}

In [ ]:
df = pd.read_csv('../../Data/Amazon.csv')
df.head()

## Creación de columnas previas al modelado

In [ ]:
df_analisis = df_cleaned[~df_cleaned["OrderStatus"].isin(['Pending', 'Returned', 'Shipped'])].copy()
df_analisis["target"] = np.where(df_analisis["OrderStatus"] == 'Cancelled', 1, 0)

# Creación de la columna 'Month'
df_analisis['Month'] = pd.to_datetime(df_analisis['OrderDate']).dt.month

# Creación de la columna 'Day'
df_analisis['Day'] = pd.to_datetime(df_analisis['OrderDate']).dt.dayofweek

# Creación de la columna 'Product_Value' basada en cuantiles de 'UnitPrice' 
df_analisis['Product_Value'] = (pd.qcut(df_analisis['UnitPrice'], q=3, labels=[1, 2, 3])) # 1: Económico, 2: Estándar, 3: Premium
df_analisis['Product_Value'] = df_analisis['Product_Value'].astype(int) 

## Transformación y preparación de datos

In [ ]:
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False, drop='first')
categorical_cols = ['Country', 'PaymentMethod', 'State', 'Category']
numeric_features = ['target','Product_Value', 'Day', 'Quantity']

encoded_data = encoder.fit_transform(df_analisis[categorical_cols])
enconded_df = pd.DataFrame(encoded_data, columns=encoder.get_feature_names_out(categorical_cols),index=df_analisis.index)

final_df = pd.concat(
    [enconded_df, df_analisis[numeric_features]],
    axis=1
)
final_df.shape

Para la etapa de transformación, hemos optado por el método **OneHotEncoder**. A diferencia de la codificación anterior, esta técnica garantiza que no se asigne un orden jerárquico artificial a las categorías, tratando a todos los valores con la misma importancia. No obstante, dado que una expansión excesiva de columnas podría aumentar el riesgo de sobreajuste (overfitting) y elevar considerablemente los tiempos de entrenamiento, hemos decidido utilizar únicamente las variables que obtuvieron las mejores puntuaciones de relevancia. Este enfoque inicial nos permite mantener un modelo equilibrado, dejando abierta la posibilidad de ajustar la selección de variables en caso de que los resultados no alcancen el rendimiento esperado.

Con los datos ya transformados, el siguiente paso consistirá en realizar la separación de las variables predictoras y la variable objetivo (target). Posteriormente, dividiremos el conjunto en datos de entrenamiento y de prueba; este proceso es fundamental para evaluar la capacidad del modelo de generalizar sus predicciones ante información nueva y asegurar que el sistema de detección de cancelaciones sea realmente efectivo y confiable antes de su implementación.

In [ ]:
X = final_df.drop('target', axis=1)
y = final_df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print(f"Distribución original en entrenamiento: {Counter(y_train)}")
print(f"Distribución original en test: {Counter(y_test)}")

Tras realizar la partición de los datos en conjuntos de entrenamiento y prueba, se confirmó el desbalance que habíamos identificado previamente. La proporción de pedidos cancelados es significativamente menor en comparación con los pedidos entregados, una situación común en este tipo de análisis que podría causar que el modelo aprenda a predecir con mayor facilidad la clase mayoritaria e ignore los casos de cancelación.

Para solucionar este inconveniente, aplicaremos una técnica de sobremuestreo denominada **SMOTE**. El objetivo de este método es generar datos sintéticos de la clase minoritaria hasta lograr que ambos tipos de pedidos estén al mismo nivel; de esta manera, garantizamos que el modelo reciba la misma cantidad de información para ambas categorías, permitiéndole aprender de forma equitativa a identificar tanto las entregas exitosas como las cancelaciones

## Balanceo de clases

In [ ]:
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print(f"Distribución sin balanceo en entrenamiento: {Counter(y_train)}")
print(f"Distribución balanceada en entrenamiento: {Counter(y_train_res)}")

Tras aplicar la técnica de sobremuestreo, las clases en el conjunto de entrenamiento presentan ahora una distribución equitativa. Es fundamental destacar que este proceso se realizó exclusivamente sobre los datos de entrenamiento para evitar la filtración de datos (data leakage); si balanceáramos también el conjunto de prueba, los resultados de evaluación serían artificialmente positivos, ya que el modelo no estaría aprendiendo a generalizar, sino simplemente a reconocer patrones de datos sintéticos que ya conoce. No obstante, somos conscientes de que generar una gran cantidad de registros para la clase de cancelaciones puede introducir ruido en el modelo, por lo que, si las métricas de precisión o la curva ROC resultan bajas, consideraremos implementar métodos alternativos de balanceo.

Por consiguiente, iniciaremos la fase de modelado comparando tres de los algoritmos de clasificación más robustos: Random Forest, XGBoost y CatBoost. Al implementar ambas arquitecturas, podremos evaluar cuál de ellas se adapta mejor a la complejidad de nuestras variables y cuál ofrece un rendimiento superior en la detección de pedidos cancelados. Este análisis comparativo será clave para identificar la herramienta más precisa y confiable antes de proceder con el ajuste final de hiperparámetros.

## Creación y evaluación de los modelos

### Modelo Random Forest Classifier

Para la construcción del modelo **Random Forest**, iniciamos con un proceso de ajuste de hiperparámetros mediante `param_grid` para identificar la configuración que mejor se adaptaba a nuestros datos. Una vez obtenidos los parámetros óptimos, definimos un nuevo diccionario con estos valores específicos; esta estrategia permite que el notebook se ejecute de manera ágil en futuras sesiones sin necesidad de repetir la búsqueda intensiva, optimizando así los tiempos de procesamiento sin sacrificar la precisión del modelo.

Despues, procederemos a evaluar el desempeño del modelo utilizando el `classification_report` y una función personalizada que se encargará de registrar los valores de **AUC-ROC** y **F1-score** directamente en un archivo CSV ubicado en la carpeta "Data". Este método de almacenamiento permitirá que cada nuevo modelo agregue sus resultados de forma automática al archivo existente, creando así un registro centralizado y persistente que facilitará la comparación final de todos los experimentos realizados. Asimismo, analizaremos la matriz de confusión y la gráfica de importancia de variables; esta última nos servirá como guía para identificar las características que menos aportan al modelo, con el fin de eliminarlas en una etapa posterior y observar si esta simplificación mejora los resultados del entrenamiento.

In [ ]:
# Puede tardarse un poco en ejecutarse
rf = RandomForestClassifier(class_weight=None, random_state=42)
# param_grid = {'n_estimators': [100, 200],
#               'max_depth': [10, None],
#               'min_samples_split': [2, 5]
# }
param_grid = {'n_estimators': [200],
              'max_depth': [None],
              'min_samples_split': [5]
}
grid_search = GridSearchCV(estimator = rf, param_grid = param_grid, cv = 5, scoring = 'f1')
grid_search.fit(X_train_res, y_train_res)

print(f"Mejores parámetros: {grid_search.best_params_}")

In [ ]:
best_rf = grid_search.best_estimator_

# Predecimos sobre el set de prueba para evaluar el rendimiento real
y_pred = best_rf.predict(X_test)

print("--- REPORTE DE CLASIFICACIÓN FINAL ---")
print(classification_report(y_test, y_pred))
evaluar_y_guardar_csv(best_rf, X_test, y_test, 'RandomForest v1.0', nombre_archivo='metricas_modelos_clasificacion_1.csv')

plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Éxito (0)', 'Falla (1)'])
disp.plot(cmap='Blues_r', values_format='d')
plt.title('Matriz de Confusión')
plt.show()

importances_clf = pd.DataFrame({
    'Variable': X.columns,
    'Importance': best_rf.feature_importances_
}).sort_values(by="Importance", ascending=False)

sns.barplot( data=importances_clf.head(20), x='Importance', y='Variable', hue = 'Variable', palette='Blues_r')
plt.title('Random Forest - Clasificación')
plt.show()

### Modelo XGBoost

Para el modelo de **XGBoost**, seguiremos una estrategia distinta para el manejo de los datos, enfocándonos en la configuración de parámetros que permitan al algoritmo gestionar el desbalance de clases de forma interna. Al ajustar el parámetro de balanceo (como `scale_pos_weight`), el modelo otorga mayor importancia a los pedidos cancelados sin necesidad de recurrir a los datos generados por SMOTE; esto nos permite trabajar con la distribución original de la información, buscando un aprendizaje más natural y reduciendo el riesgo de sobreajuste por datos sintéticos.

En cuanto a la evaluación, mantendremos la misma metodología aplicada en el modelo de Random Forest para asegurar una comparativa justa. Utilizaremos el reporte de clasificación y nuestra función personalizada para almacenar el **AUC-ROC** y el **F1-score** en el archivo csv que guarda todos resultados finales. Asimismo, analizaremos la matriz de confusión y la importancia de las variables para determinar si existen características que no aportan valor significativo; de ser así, procederemos a eliminarlas en una siguiente prueba para verificar si esto optimiza el rendimiento general del modelo.

In [ ]:
# 1. Calculamos el factor de escala para el desbalance
counts = y_train.value_counts()
scale_weight = counts[0] / counts[1]

# 2. Creamos el modelo XGBoost
xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_weight,
    eval_metric='logloss',
    random_state=42
)

# 3. Entrenamiento
xgb_model.fit(X_train, y_train)

# 4. Predicción
y_pred_xgb = xgb_model.predict(X_test)
y_probs_xgb = xgb_model.predict_proba(X_test)[:, 1]

# 5. Evaluación
print("--- REPORTE DE CLASIFICACIÓN: XGBOOST ---")
print(classification_report(y_test, y_pred_xgb))
evaluar_y_guardar_csv(xgb_model, X_test, y_test, 'XGBoost v1.0', nombre_archivo='metricas_modelos_clasificacion_1.csv')

# 6. Matriz de Confusión
cm = confusion_matrix(y_test, y_pred_xgb)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Éxito (0)', 'Falla (1)'])
disp.plot(cmap='Blues')

### Modelo CatBoost

In [ ]:
X = df_analisis[['Country', 'PaymentMethod', 'State', 'Category', 'Product_Value', 'Day', 'Quantity']]
y = df_analisis['target']

X_train_cat, X_test_cat, y_train_cat, y_test_cat = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

categorical_features = ['Country', 'PaymentMethod', 'State', 'Category', 'Product_Value', 'Day', 'Quantity']
# numeric_features = ['target', 'Product_Value', 'Day', 'Quantity']

# 1. Configurar el modelo
cat_model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.1,
    depth=6,
    auto_class_weights='Balanced', # Para manejar el desbalance de clases
    loss_function='Logloss',
    verbose=100,
    allow_writing_files=False
)

# 2. Entrenar el modelo
cat_model.fit(
    X_train_cat, y_train_cat,
    cat_features=categorical_features, 
    eval_set=(X_test_cat, y_test_cat),
    plot=False
)

# 3. Predicciones y Evaluación
y_pred_cat = cat_model.predict(X_test_cat)
y_probs_cat = cat_model.predict_proba(X_test_cat)[:, 1]

print("\n--- REPORTE DE CLASIFICACIÓN: CATBOOST ---")
print(classification_report(y_test_cat, y_pred_cat))
evaluar_y_guardar_csv(cat_model, X_test_cat, y_test_cat, 'CatBoost v1.0',  nombre_archivo='metricas_modelos_clasificacion_1.csv')

# 5. Matriz de Confusión
cm = confusion_matrix(y_test_cat, y_pred_cat)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Matriz de Confusión - CatBoost')
plt.ylabel('Real')
plt.xlabel('Predicho')
plt.show()